## Setup

In [2]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets

In [3]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

## Imports
import os
import sys
import nltk
import time
import torch
import random
import subprocess
import numpy as np
import pandas as pd
import datetime as dt
from itertools import groupby
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import pipeline
from collections import Counter, defaultdict
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report as seq_classification
from sklearn.metrics import f1_score as skl_f1, precision_score as skl_precision, recall_score as skl_recall, classification_report as skl_classification

Mounted at /content/drive/


In [4]:
# Append the library files into the notebook system path for import
sys.path.append('/content/drive/Shareddrives/Machine Translation/Model benchmarking/Libraries/1.0.2')
# import custom library files
import ner, utils

## Load datasets

### Peoples daily
https://huggingface.co/datasets/peoples_daily_ner

In [5]:
pdaily_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6,
}

pdaily = ner.ReadNERData()
pdaily_words, pdaily_labels = pdaily.read_dataset('peoples_daily_ner', pdaily_label_map)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:72: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test Split


  0%|          | 0/4637 [00:00<?, ?it/s]

In [6]:
print(ner.check_labels(pdaily_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC

# harem_label_alignment = {
#     'B-COISA': 'O',
#     'B-LOCAL': 'B-LOC',
#     'B-OBRA': 'O',
#     'I-VALOR': 'O',
#     'I-OUTRO': 'O',
#     'B-ABSTRACCAO': 'O',
#     'B-TEMPO': 'O',
#     'O': 'O',
#     'I-TEMPO': 'O',
#     'B-ACONTECIMENTO': 'O',
#     'I-ORGANIZACAO': 'I-ORG',
#     'I-PESSOA': 'I-PER',
#     'B-PESSOA': 'B-PER',
#     'B-VALOR': 'O',
#     'I-ABSTRACCAO': 'O',
#     'B-ORGANIZACAO': 'B-ORG',
#     'I-COISA': 'O',
#     'I-LOCAL': 'I-LOC',
#     'B-OUTRO': 'O',
#     'I-ACONTECIMENTO': 'O',
#     'I-OBRA': 'O',
# }

# Align the dataset labels to the standard labels
# harem_labels = ner.align_dataset(harem_labels, harem_label_alignment)
# print(ner.check_labels(harem_labels))

{'I-ORG', 'I-LOC', 'O', 'B-LOC', 'I-PER', 'B-PER', 'B-ORG'}


### wikiann

In [8]:
wikiann_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6
}

wikiann = ner.ReadNERData()
wikiann_words, wikiann_labels = wikiann.read_dataset('wikiann', wikiann_label_map, lang='zh')

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/10000 [00:00<?, ?it/s]

In [ ]:
print(ner.check_labels(wikiann_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC

{'I-LOC', 'O', 'I-PER', 'B-PER', 'B-ORG', 'I-ORG', 'B-LOC'}


### MRSA
https://huggingface.co/datasets/msra_ner


In [9]:
mrsa_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6,
}

mrsa = ner.ReadNERData()
mrsa_words, mrsa_labels = mrsa.read_dataset('msra_ner', mrsa_label_map)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:72: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test Split


  0%|          | 0/3443 [00:00<?, ?it/s]

# Evaluate model

In [10]:
alignment = {
'B-LOC': 'B-LOC',
'B-MISC': 'O',
'B-ORG': 'B-ORG',
'I-LOC': 'I-LOC',
'I-MISC': 'O',
'I-ORG': 'I-ORG',
'I-PER': 'I-PER',
'O': 'O'
}

model_name = "xlm-roberta-large-finetuned-conll03-english"
model_name_output = 'xlm-roberta-large'
model_evaluation = ner.ModelEvaluation(
    model_name,
    alignment
)

config.json:   0%|          | 0.00/852 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of the model checkpoint at xlm-roberta-large-finetuned-conll03-english were not used when initializing XLMRobertaForTokenClassification: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
- This IS expected if you are initializing XLMRobertaForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
# model_evaluation.model.config.id2label

{0: 'B-LOC',
 1: 'B-MISC',
 2: 'B-ORG',
 3: 'I-LOC',
 4: 'I-MISC',
 5: 'I-ORG',
 6: 'I-PER',
 7: 'O'}

### Peoples daily

In [11]:
data_name = "peoples_daily_ner"
pdaily_evaluation_output = model_evaluation.evaluate_model(pdaily_words, pdaily_labels)

  0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classif

In [12]:
pdaily_seqeval = pdaily_evaluation_output.get_classification('Seqeval')
pdaily_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.3417,0.2419,0.2833,3559
1,ORG,0.2060,0.1281,0.1580,2185
2,PER,0.4927,0.4887,0.4907,1864
3,micro,0.3582,0.2697,0.3077,7608
4,macro,0.3468,0.2863,0.3107,7608
5,weighted,0.3397,0.2697,0.2981,7608


In [13]:
pdaily_sklearn = pdaily_evaluation_output.get_classification('Sklearn')
pdaily_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.0000,0.0000,0.0000,3559
1,B-ORG,0.0000,0.0000,0.0000,2185
2,B-PER,0.0000,0.0000,0.0000,1864
3,I-LOC,0.4737,0.4756,0.4746,4819
4,I-ORG,0.7375,0.3399,0.4653,8756
5,I-PER,0.6759,0.8103,0.7371,3601
6,O,0.9398,0.9962,0.9672,193205
7,accuracy,0.9205,217989,None,None
8,macro,0.4039,0.3746,0.3777,217989
9,weighted,0.8842,0.9205,0.8986,217989


### wikiann

In [14]:
data_name = "wikiann"
wikiann_evaluation_output = model_evaluation.evaluate_model(wikiann_words, wikiann_labels)

  0%|          | 0/625 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classif

In [15]:
wikiann_seqeval = wikiann_evaluation_output.get_classification('Seqeval')
wikiann_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.1964,0.2982,0.2368,4474
1,ORG,0.1815,0.1021,0.1307,4114
2,PER,0.3085,0.4072,0.3510,3939
3,micro,0.2347,0.2681,0.2503,12527
4,macro,0.2288,0.2692,0.2395,12527
5,weighted,0.2268,0.2681,0.2379,12527


In [16]:
wikiann_sklearn = wikiann_evaluation_output.get_classification('Sklearn')
wikiann_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.0000,0.0000,0.0000,4371
1,B-ORG,0.0000,0.0000,0.0000,3778
2,B-PER,0.0000,0.0000,0.0000,3895
3,I-LOC,0.3359,0.5015,0.4023,12282
4,I-ORG,0.3698,0.1718,0.2346,17394
5,I-PER,0.4559,0.4696,0.4627,12887
6,O,0.8366,0.9192,0.8759,151044
7,accuracy,0.7490,205651,None,None
8,macro,0.2854,0.2946,0.2822,205651
9,weighted,0.6943,0.7490,0.7162,205651


### MRSA

In [17]:
data_name = "msra_ner"
mrsa_evaluation_output = model_evaluation.evaluate_model(mrsa_words, mrsa_labels)

  0%|          | 0/216 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classif

In [18]:
mrsa_seqeval = mrsa_evaluation_output.get_classification('Seqeval')
mrsa_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.3280,0.1967,0.2460,2826
1,ORG,0.2091,0.1437,0.1704,1308
2,PER,0.4603,0.4522,0.4562,1411
3,micro,0.3472,0.2492,0.2902,5545
4,macro,0.3325,0.2642,0.2908,5545
5,weighted,0.3336,0.2492,0.2816,5545


In [19]:
mrsa_sklearn = mrsa_evaluation_output.get_classification('Sklearn')
mrsa_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.0000,0.0000,0.0000,2826
1,B-ORG,0.0000,0.0000,0.0000,1308
2,B-PER,0.0000,0.0000,0.0000,1411
3,I-LOC,0.5462,0.4028,0.4637,4317
4,I-ORG,0.7099,0.3608,0.4784,5507
5,I-PER,0.6855,0.7973,0.7372,2748
6,O,0.9411,0.9969,0.9682,150722
7,accuracy,0.9250,168839,None,None
8,macro,0.4118,0.3654,0.3782,168839
9,weighted,0.8884,0.9250,0.9037,168839
